## Step 1: Setup

### Intent
The goal of this step is to prepare the working environment for fine-tuning GPT-2 on the WikiText-103 dataset.  
This includes installing the required libraries and importing the necessary modules.

### AI Prompt
Help me set up a Google Colab notebook for fine-tuning GPT-2 on WikiText-103 using Hugging Face Transformers and Datasets."

### Review & Modify
The AI suggested installing the main Hugging Face libraries needed for the project, such as transformers and datasets.

We added the -q flag to reduce installation logs and keep the notebook clean.

Since our final implementation uses a manual PyTorch training loop instead of Hugging Face Trainer, we also imported the required PyTorch tools, such as DataLoader, AdamW, torch, and tqdm.

We also mounted Google Drive so the fine-tuned model and checkpoints could be saved directly to Drive.

In [ ]:
# Install required libraries
!pip install transformers datasets evaluate accelerate -q

# Import basic libraries
import math
import torch
import numpy as np

# Import Hugging Face tools
from datasets import load_dataset
from transformers import (
    GPT2Tokenizer,                  # Converts text into tokens that GPT-2 can understand
    GPT2LMHeadModel,                # Loads the GPT-2 model for language modeling
    DataCollatorForLanguageModeling,# Prepares batches for causal language modeling
    pipeline                        # Used for easy text generation
)

# Import PyTorch training tools
from torch.utils.data import DataLoader  # Divides the dataset into small batches
from torch.optim import AdamW            # Optimizer that updates the model weights
from tqdm import tqdm                    # Shows a progress bar during training


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
# Google Drive
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Data Pipeline

### Intent
We need to load the WikiText-103 dataset and prepare it for GPT-2 fine-tuning. Since GPT-2 works with tokens, we tokenize the raw text using the GPT-2 tokenizer.

### AI Prompt
"Show me how to load WikiText-103 from Hugging Face datasets and tokenize it for GPT-2 language modeling."

### Review & Modify
The AI suggested loading the full dataset directly. We modified this by selecting a smaller subset because the full WikiText-103 dataset is too large for our Colab resources.

We also added a `.filter()` step to remove empty text rows, since the raw dataset contains many blank lines that can waste computation or cause evaluation issues.

For tokenization, we made three GPT-2-specific adjustments:
1. Set `tokenizer.pad_token = tokenizer.eos_token` because GPT-2 has no default padding token so we add eos_token in the end.
2. Used `DataCollatorForLanguageModeling` with `mlm=False` because GPT-2 predicts the next token, not masked tokens.
3. Added `remove_columns=["text"]` after tokenization to avoid tensor conversion issues.

In [ ]:
# Load WikiText-103 raw dataset from Hugging Face
dataset = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
dataset = dataset.filter(lambda example: example["text"].strip() != "")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 2891
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1165029
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 2461
    })
})


In [ ]:
# Use a smaller subset for practical training in Colab
train_data = dataset["train"].select(range(50000))
valid_data = dataset["validation"].select(range(2000))

print("Training samples:", len(train_data))
print("Validation samples:", len(valid_data))

Training samples: 50000
Validation samples: 2000


In [ ]:
# Load GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 does not have a default padding token, so we use EOS as padding
tokenizer.pad_token = tokenizer.eos_token

# Instead of DataCollatorForLanguageModeling, use a custom or default approach,
# or ensure you map labels where pad_token_id is set to -100.
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    # The model calculates loss using 'labels'.
    # We set padding tokens to -100 so they are ignored by the PyTorch CrossEntropyLoss.
    tokens["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in tokens["input_ids"]
    ]
    return tokens

#example
#input_ids = [10, 25, 80, 50256, 50256]
#labels    = [10, 25, 80, -100, -100]
# Tokenize train and validation sets
tokenized_train = train_data.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_valid = valid_data.map(tokenize_function, batched=True, remove_columns=["text"])

# Data collator for causal language modeling
# mlm=False because GPT-2 is not a masked language model
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # GPT-2 uses causal language modeling, not masked language modeling
)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

## Step 3: Model

### Intent
We need to load the pre-trained GPT-2 model. GPT-2 is suitable for creative text generation because it is a decoder-only Transformer trained for next-token prediction.

### AI Prompt
"Explain how to load GPT-2 for causal language modeling".

### Review & Modify

The AI suggested loading GPT-2 using `GPT2LMHeadModel.from_pretrained("gpt2")`, together with a general explanation of decoder-only Transformer models.

We also added `model.resize_token_embeddings(len(tokenizer))` to ensure the model’s embedding layer is aligned with the tokenizer size. In our case, we used the existing EOS token as the padding token, so the vocabulary size did not change, but this line keeps the setup consistent and safe if tokenizer tokens are updated.

In [ ]:
# Load pre-trained GPT-2 model
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Resize token embeddings because we changed the tokenizer padding token
model.resize_token_embeddings(len(tokenizer))

# Move model to GPU if available
model.to(device)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)



## Step 4: Training

### Intent
Fine-tune GPT-2 on a subset of WikiText-103 so the model adapts better to Wikipedia-style text generation.

### AI Prompt
"Fine-tune GPT-2 on tokenized WikiText-103 data"

### Review & Modify
The AI first suggested using Hugging Face’s Trainer, but we decided to replace it with a manual training loop using for epoch.

We modified the code for our Colab environment by using a small batch size of 8, training for 2 epochs, and using DataLoader for both training and validation data.

We also added manual evaluation using model.eval() and torch.no_grad() to calculate validation loss and perplexity. Finally, we saved the fine-tuned model and tokenizer to Google Drive after each epoch using save_pretrained(), and also saved a checkpoint with the model and optimizer states.

In [ ]:
import os
import re
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from torch.utils.data import DataLoader

# Create DataLoaders for batching the tokenized datasets
train_loader = DataLoader(
    tokenized_train,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

valid_loader = DataLoader(
    tokenized_valid,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator
)

base_save_dir = "/content/drive/MyDrive/gpt2ProjectAfter"
num_epochs = 7

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Find latest saved checkpoint
latest_epoch = 0
latest_checkpoint_path = None

if os.path.exists(base_save_dir):
    for folder_name in os.listdir(base_save_dir):
        match = re.match(r"gpt2_wikitext103_epoch-(\d+)", folder_name)
        if match:
            epoch_num = int(match.group(1))
            checkpoint_path = os.path.join(base_save_dir, folder_name, "checkpoint.pt")
            if os.path.exists(checkpoint_path) and epoch_num > latest_epoch:
                latest_epoch = epoch_num
                latest_checkpoint_path = checkpoint_path

# Resume training from the latest checkpoint if available
if latest_checkpoint_path is not None:
    print(f"Resuming from checkpoint: Epoch {latest_epoch}")

    model_path = os.path.join(base_save_dir, f"gpt2_wikitext103_epoch-{latest_epoch}")

    model = GPT2LMHeadModel.from_pretrained(model_path)
    tokenizer = GPT2Tokenizer.from_pretrained(model_path)

    model.to(device)

    optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

    # Restore optimizer state so training continues properly
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = latest_epoch

else:
    print("No checkpoint found. Starting training from scratch.")

    model.to(device)
    optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

    start_epoch = 0

# Continue training from the correct epoch
for epoch in range(start_epoch, num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    model.train()
    total_train_loss = 0

    for batch in tqdm(train_loader, desc="Training"):
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward pass: compute predictions and loss
        outputs = model(**batch)
        loss = outputs.loss

        # Backward pass and weight update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Training Loss: {avg_train_loss:.4f}")

    # Save model, tokenizer, and checkpoint after each epoch
    save_path = f"{base_save_dir}/gpt2_wikitext103_epoch-{epoch + 1}"
    os.makedirs(save_path, exist_ok=True)

    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": avg_train_loss
    }, f"{save_path}/checkpoint.pt")

    print(f"Model saved to {save_path}")

Resuming from checkpoint: Epoch 7


Loading weights:   0%|          | 0/148 [00:01<?, ?it/s]

## Step 5: Evaluation

### Intent
We evaluate the model using validation loss and perplexity. Perplexity is a common metric for language modeling. Lower perplexity means the model predicts the next tokens better.

### AI Prompt
"How can I evaluate a fine-tuned GPT-2 model using validation loss and perplexity?"

### Review & Modify
The AI initially suggested using trainer.evaluate(), but since we replaced Hugging Face Trainer with a manual PyTorch loop, we modified the evaluation code.

Instead of retraining the model, we loaded the saved checkpoints from Epoch 1 to Epoch 2. For each checkpoint, we used model.eval() and torch.no_grad() to run the model on the validation data without updating its weights.

Then, we calculated the average validation loss for each checkpoint and converted it to perplexity using math.exp(avg_eval_loss). We also loaded the saved training loss from each checkpoint file.

This allowed us to compare the training loss, validation loss, and perplexity across all four epochs. The results showed that Epoch 1 achieved the best validation performance, while later epochs showed increasing validation loss and perplexity, indicating overfitting.

Since creative text generation cannot be fully evaluated using automatic metrics alone, we use validation loss and perplexity as quantitative metrics, and later combine them with generated text examples and error analysis.

In [ ]:
import torch
import math
import pandas as pd
from tqdm import tqdm
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Store evaluation results for each checkpoint
results = []

# Evaluate saved checkpoints from Epoch 1 to Epoch 10
for epoch in range(1, 8):
    model_path = f"/content/drive/MyDrive/gpt2ProjectAfter/gpt2_wikitext103_epoch-{epoch}"

    # Load the saved model and tokenizer for this epoch
    model = GPT2LMHeadModel.from_pretrained(model_path)
    tokenizer = GPT2Tokenizer.from_pretrained(model_path)

    model.to(device)
    # Set model to evaluation mode
    model.eval()

    total_eval_loss = 0

    # Disable gradient calculation during evaluation
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc=f"Validation Epoch {epoch}"):
            batch = {k: v.to(device) for k, v in batch.items()}

            # Forward pass on validation data
            outputs = model(**batch)
            loss = outputs.loss

            total_eval_loss += loss.item()

    # Calculate average validation loss
    avg_eval_loss = total_eval_loss / len(valid_loader)

    # Convert validation loss to perplexity
    perplexity = math.exp(avg_eval_loss)

    # Load the saved training loss from checkpoint
    checkpoint_path = f"{model_path}/checkpoint.pt"
    checkpoint = torch.load(checkpoint_path, map_location=device)
    train_loss = checkpoint["train_loss"]

    results.append({
        "epoch": epoch,
        "training_loss": train_loss,
        "validation_loss": avg_eval_loss,
        "perplexity": perplexity
    })

# Convert results to a table
results_df = pd.DataFrame(results)
results_df

Loading weights:   0%|          | 0/148 [00:01<?, ?it/s]

Validation Epoch 1: 100%|██████████| 250/250 [00:23<00:00, 10.77it/s]


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Validation Epoch 2: 100%|██████████| 250/250 [00:23<00:00, 10.74it/s]


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Validation Epoch 6: 100%|██████████| 250/250 [00:23<00:00, 10.75it/s]


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Validation Epoch 7: 100%|██████████| 250/250 [00:23<00:00, 10.72it/s]


,epoch,training_loss,validation_loss,perplexity
0,1,3.442433,3.368892,29.046324
1,2,2.844431,3.433180,30.975000
2,3,2.713325,3.472214,32.207959
3,4,2.592029,3.523298,33.896040
4,5,2.479704,3.580508,35.891759
5,6,2.372280,3.652393,38.566857
6,7,2.272566,3.717625,41.166512


## Step 6: Text Generation

### Intent
We test the fine-tuned GPT-2 model by giving it prompts and generating Wikipedia-style text.

### AI Prompt
"Show me how to generate text from a fine-tuned GPT-2 model using top-k, top-p, and temperature."

### Review & Modify
The AI initially suggested using the Hugging Face `pipeline` for text generation. However, the pipeline caused generation warnings because of conflicts between default generation settings and our custom parameters.

We modified the code to use `model.generate()` directly. This gives us clearer control over generation parameters such as `max_new_tokens`, `top_k`, `top_p`, `temperature`, and `repetition_penalty`.

We also added `pad_token_id=tokenizer.eos_token_id` because GPT-2 does not have a default padding token.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
# @title Interactive GPT-2 Creative Text Generation
# @markdown Enter a starting prompt and adjust the creativity parameters below.

prompt_text = "The history of artificial intelligence" # @param {type:"string"}
max_tokens = 120 # @param {type:"slider", min:20, max:300, step:10}
temperature = 1 # @param {type:"slider", min:0.1, max:2.0, step:0.1}
top_k = 60 # @param {type:"slider", min:1, max:100, step:1}
top_p = 0.9 # @param {type:"slider", min:0.1, max:1.0, step:0.05}
repetition_penalty = 1.2 # @param {type:"slider", min:1.0, max:2.0, step:0.1}

print("=" * 80)
print(f"Prompt: {prompt_text}")
print(f"Creativity (Temperature): {temperature} | Top-P: {top_p}")
print("-" * 80)

epoch_path = "/content/drive/MyDrive/gpt2ProjectAfter/gpt2_wikitext103_epoch-1"

tokenizer = GPT2Tokenizer.from_pretrained(epoch_path)
model = GPT2LMHeadModel.from_pretrained(epoch_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Tokenize the input from the text box
inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

# Generate text dynamically using the slider variables
output_ids = model.generate(
    **inputs,
    max_new_tokens=max_tokens,
    do_sample=True,
    top_k=top_k,
    top_p=top_p,
    temperature=temperature,
    repetition_penalty=repetition_penalty,
    pad_token_id=tokenizer.eos_token_id
)

# Decode and print the result
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(generated_text)
print("=" * 80)

Prompt: The history of artificial intelligence
Creativity (Temperature): 1 | Top-P: 0.9
--------------------------------------------------------------------------------


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The history of artificial intelligence is a long one , as the computer becomes increasingly sophisticated . AI researchers are interested in how to use its powers against humans , and human cooperation with AI researchers is an important part that makes it so that individuals will have access both their own intelligence ( and a common sense ) versus those acquired through experimentation in machine learning algorithms . 
One area that can be explored is how to create neural networks based on information generated by machines such computers .  This has not been used yet but some projects may develop useful techniques for developing systems using machine learning or artificial intelligences from new data .  In 2008 , the


## Baseline Comparison

To evaluate the effect of fine-tuning, we compare our fine-tuned GPT-2 model with the original pre-trained GPT-2 model using the same prompt and generation settings.

In [ ]:
# Baseline comSparison: original GPT-2 vs fine-tuned GPT-2

baseline_model_name = "gpt2"

baseline_tokenizer = GPT2Tokenizer.from_pretrained(baseline_model_name)
baseline_model = GPT2LMHeadModel.from_pretrained(baseline_model_name)

baseline_tokenizer.pad_token = baseline_tokenizer.eos_token

baseline_model.to(device)
baseline_model.eval()

# Same prompt and generation settings
prompt_text = "The history of artificial intelligence"

max_tokens = 120
temperature = 0.8
top_k = 50
top_p = 0.9
repetition_penalty = 1.1

print("=" * 80)
print("Prompt:", prompt_text)
print("=" * 80)

# ---------- Baseline GPT-2 ----------
baseline_inputs = baseline_tokenizer(prompt_text, return_tensors="pt").to(device)

with torch.no_grad():
    baseline_output_ids = baseline_model.generate(
        **baseline_inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        repetition_penalty=repetition_penalty,
        pad_token_id=baseline_tokenizer.eos_token_id
    )

baseline_text = baseline_tokenizer.decode(
    baseline_output_ids[0],
    skip_special_tokens=True
)

print("\nBaseline GPT-2 Output:")
print("-" * 80)
print(baseline_text)


from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# Load Fine-tuned GPT-2 from Epoch 1
epoch_path = "/content/drive/MyDrive/gpt2ProjectAfter/gpt2_wikitext103_epoch-1"

tokenizer = GPT2Tokenizer.from_pretrained(epoch_path)
model = GPT2LMHeadModel.from_pretrained(epoch_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ---------- Fine-tuned GPT-2 - Epoch 1 ----------
model.eval()

fine_tuned_inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

with torch.no_grad():
    fine_tuned_output_ids = model.generate(
        **fine_tuned_inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        top_k=top_k,
        top_p=top_p,
        temperature=temperature,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id
    )

fine_tuned_text = tokenizer.decode(
    fine_tuned_output_ids[0],
    skip_special_tokens=True
)

print("\nFine-tuned GPT-2 Epoch 1 Output:")
print("-" * 80)
print(fine_tuned_text)
print("=" * 80)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Prompt: The history of artificial intelligence

Baseline GPT-2 Output:
--------------------------------------------------------------------------------
The history of artificial intelligence has been documented for many decades and is well known to those who work in this field. Some researchers have claimed that AI may not even be able get past the "human mind," but some studies suggest it could eventually become capable, with little effort from any human being or machines at all (like Einstein) — making him/her a unique technology worthy enough by itself to challenge conventional wisdom about our mental capacities as we learn more everyday over time."
"In other words: Artificial Intelligence's potential here can't simply come together into something like an 'optical machine,' which would then create its own


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Fine-tuned GPT-2 Epoch 1 Output:
--------------------------------------------------------------------------------
The history of artificial intelligence is an important topic . AI researchers have developed a wide range in their research , many with very different perspectives . Many of these viewpoints are based on the assumption that we can think and act like machines , or that computers need to be able to do things humans wouldn 't want to do . 
 = Cognitive science is a field which includes several disciplines such as Artificial Intelligence ( AI ) , Computer Vision ( CVA ) , Machine Learning ( ML ) , and Computer Vision at least one other field , Cognitive Computing , which includes Deep Neural Networks , Deep Belief Machines , Deep learning techniques , Deep learning ,


## Training Results Summary

GPT-2 was fine-tuned on a subset of WikiText-103 using a batch size of 8.

The results show that the training loss decreased from **3.4424** to **2.2726**, which means that the model continued learning from the training data.

However, the best validation result was achieved after **Epoch 1**, with a validation loss of **3.3689** and a perplexity of **29.0463**.

After Epoch 1, the validation loss and perplexity increased gradually. By Epoch 7, the validation loss reached **3.7176** and the perplexity reached **41.1665**. This indicates that the model started to overfit the training data.

Therefore, **Epoch 1 was selected as the best checkpoint**.

| Epoch | Training Loss | Validation Loss | Perplexity |
| ----- | ------------: | --------------: | ---------: |
| 1     |        3.4424 |          3.3689 |    29.0463 |
| 2     |        2.8444 |          3.4332 |    30.9750 |
| 3     |        2.7133 |          3.4722 |    32.2080 |
| 4     |        2.5920 |          3.5233 |    33.8960 |
| 5     |        2.4797 |          3.5805 |    35.8918 |
| 6     |        2.3723 |          3.6524 |    38.5669 |
| 7     |        2.2726 |          3.7176 |    41.1665 |


## Step 7: Error Analysis

### Intent
Analyze the generated outputs and identify the main weaknesses of the fine-tuned GPT-2 model.

### AI Prompt
"Help me write an error analysis for GPT-2 text generation results on WikiText-103."

### Review & Modify
The AI suggested general possible errors for GPT-2 generation. We modified the analysis according to the behavior observed in the outputs generated by our fine-tuned model.

### Error Analysis

The generated outputs show that the fine-tuned GPT-2 model learned some Wikipedia-like writing patterns. The model often produces longer and more formal text, uses topic-related vocabulary, and begins with a relevant introduction.

However, several limitations were observed:

1. **Factual inaccuracies**  
   The model sometimes generates information that sounds realistic but is not necessarily correct. This happens because GPT-2 predicts the next token based on learned patterns, but it does not verify facts.

2. **Hallucinated details**  
   The model may generate names, dates, theories, or relationships that appear academic but are not supported by reliable evidence.

3. **Weak coherence**  
   Some generated texts begin with a relevant idea but gradually become unclear or move away from the original topic.

4. **WikiText formatting artifacts**  
   The model sometimes produces formatting patterns from the WikiText-103 dataset, such as `@-@` or section-style markers. This shows that the model learned not only the writing style, but also some raw formatting patterns from the dataset.

5. **Effect of decoding parameters**  
   Higher creativity settings, such as high temperature, can increase diversity in the generated text, but they may also reduce coherence and factual reliability.

Overall, the fine-tuned model improved the writing style and produced more Wikipedia-like text. However, the generated content still requires human review because the model may produce factual errors, hallucinated details, weak coherence, or formatting artifacts.



## Reflection: AI as Coding Partner

### One AI Success
AI was helpful in organizing the notebook structure and building the initial GPT-2 fine-tuning pipeline. It helped us understand how to load WikiText-103, tokenize the data, load GPT-2, train the model, and evaluate it using validation loss and perplexity.

### One AI Failure
AI initially suggested code that was not fully compatible with our environment and project needs. For example, it suggested using Hugging Face `Trainer`, but we later replaced it with a manual PyTorch training loop to have more control over the training, evaluation, and saving process. We also had to fix compatibility issues, such as replacing `AdamW` from `transformers` with `AdamW` from `torch.optim`.

### Overall Assessment
Overall, AI was useful as a coding partner, but it was not enough to copy its answers directly. We had to review the code, debug errors, adjust the dataset size to fit Colab resources, remove empty text rows, save the model correctly to Google Drive, and analyze the generated outputs ourselves. AI helped speed up the process, but understanding, debugging, and final decisions were our responsibility.